# Setup

Load the three realtime tables the ingestion pipeline has collected so far.
Everything below is worked out from these three frames, with no project imports, so the logic stays visible here before it is moved into `src/`.

In [20]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)

# The notebook may be launched from either the repo root or notebooks/, so anchor on the folder that holds the database.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
conn = sqlite3.connect(PROJECT_ROOT / "data/traffic.db")

trip_updates_df = pd.read_sql("SELECT * FROM trip_updates", conn)
vehicle_positions_df = pd.read_sql("SELECT * FROM vehicle_positions", conn)
alerts_df = pd.read_sql("SELECT * FROM alerts", conn)

print(trip_updates_df.shape, vehicle_positions_df.shape, alerts_df.shape)

(7620, 11) (151, 14) (200, 7)


# Feed volume and poll cadence

How much data each feed holds, and how regularly it was actually polled.

In [21]:
rows = []
for feed, df in [("trip_updates", trip_updates_df), ("vehicle_positions", vehicle_positions_df), ("alerts", alerts_df)]:
    polls = pd.to_datetime(pd.Series(df["fetched_at"].unique()), utc=True, format="ISO8601").sort_values()
    gaps = polls.diff().dt.total_seconds().dropna()
    rows.append({
        "feed": feed,
        "n_rows": len(df),
        "n_polls": len(polls),
        "span_minutes": round((polls.max() - polls.min()).total_seconds() / 60, 1),
        "median_gap_seconds": gaps.median(),
        "max_gap_seconds": gaps.max(),
        # A feed that has been created but never written to would divide by zero here.
        "rows_per_poll": round(len(df) / len(polls), 1) if len(polls) else float("nan"),
    })

pd.DataFrame(rows)

,feed,n_rows,n_polls,span_minutes,median_gap_seconds,max_gap_seconds,rows_per_poll
0,trip_updates,7620,9,25.6,68.724024,871.346208,846.7
1,vehicle_positions,151,9,25.6,68.665710,871.375740,16.8
2,alerts,200,1,0.0,NaN,NaN,200.0


# Null rate and cardinality

Check for missing values, and see how much each column actually varies before relying on it as a feature.

In [22]:
pd.DataFrame({"null_frac": trip_updates_df.isna().mean(), "n_unique": trip_updates_df.nunique()})

,null_frac,n_unique
entity_id,0.0,185
trip_id,0.0,48
route_id,0.0,1
start_date,0.0,1
stop_sequence,0.0,21
stop_id,0.0,42
arrival_time,0.0,1424
arrival_delay,0.0,52
departure_time,0.0,1420
schedule_relationship,0.0,1


In [23]:
pd.DataFrame({"null_frac": vehicle_positions_df.isna().mean(), "n_unique": vehicle_positions_df.nunique()})

,null_frac,n_unique
entity_id,0.0,150
trip_id,0.0,25
route_id,0.0,1
vehicle_id,0.0,19
vehicle_label,0.0,19
lat,0.0,138
lon,0.0,137
bearing,0.0,113
speed,0.0,28
current_stop_sequence,0.0,20


# Putting the two clocks on one scale

The feed reports `arrival_time` as POSIX seconds while `fetched_at` is stored as an ISO string, so they need a common scale before a stop can be called past or upcoming.

In [24]:
EPOCH = pd.Timestamp("1970-01-01", tz="UTC")

# Subtracting the epoch avoids depending on how the installed pandas stores datetime precision.
def to_epoch_seconds(fetched_at):
    return (pd.to_datetime(fetched_at, utc=True, format="ISO8601") - EPOCH).dt.total_seconds().astype("int64")


# arrival_time is 0 at a trip's first stop, where there is no arrival to report.
stops_df = trip_updates_df[trip_updates_df["arrival_time"] > 0].copy()
stops_df["secs_to_arrival"] = stops_df["arrival_time"] - to_epoch_seconds(stops_df["fetched_at"])
stops_df["is_passed"] = stops_df["secs_to_arrival"] <= 0

stops_df[["trip_id", "stop_sequence", "arrival_delay", "secs_to_arrival", "is_passed"]].head()

,trip_id,stop_sequence,arrival_delay,secs_to_arrival,is_passed
1,0241-001-109-016:1000,2,0,952,False
2,0241-001-109-016:1000,3,0,1121,False
3,0241-001-109-016:1000,4,0,1271,False
4,0241-001-109-016:1000,5,0,1428,False
5,0241-001-109-016:1000,6,0,1588,False


# Observed stops against forward predictions

A single poll carries every remaining stop of a trip, so only the passed stops are measurements and the rest are the operator's own forecast.

In [25]:
summary = stops_df.groupby(stops_df["is_passed"].map({True: "passed", False: "upcoming"}))["arrival_delay"].agg(
    n_rows="size", mean_delay="mean", min_delay="min", max_delay="max"
)
summary.insert(1, "row_share", summary["n_rows"] / len(stops_df))
summary

,n_rows,row_share,mean_delay,min_delay,max_delay
is_passed,,,,,
passed,2798,0.385187,10.755540,-25,45
upcoming,4466,0.614813,3.145544,-25,38


In [26]:
buckets = pd.cut(stops_df["secs_to_arrival"], [-10**9, 0, 300, 900, 1800, 10**9], labels=["passed", "<5m", "5-15m", "15-30m", ">30m"])
stops_df.groupby(buckets, observed=True)["arrival_delay"].agg(["count", "mean", "std", "min", "max"])

,count,mean,std,min,max
secs_to_arrival,,,,,
passed,2798,10.755540,9.898642,-25,45
<5m,259,8.583012,10.797819,-25,38
5-15m,509,6.469548,10.050148,-25,33
15-30m,776,5.766753,9.723585,-25,33
>30m,2922,1.388433,5.463900,-15,33


# One delay observation per poll and trip

Averaging a whole poll would mix measurements with forecasts, so keep the freshest stop for each trip and prefer one it has already passed.

In [27]:
stops_df["secs_from_now"] = stops_df["secs_to_arrival"].abs()

# Sorting passed stops first, then by distance from now, makes the freshest stop the first row of each group.
current_df = (
    stops_df.sort_values(["is_passed", "secs_from_now"], ascending=[False, True])
    .groupby(["fetched_at", "trip_id"], as_index=False)
    .first()
)

print(current_df.shape)
current_df[["fetched_at", "trip_id", "stop_sequence", "arrival_delay", "secs_to_arrival", "is_passed"]].head()

(374, 14)


,fetched_at,trip_id,stop_sequence,arrival_delay,secs_to_arrival,is_passed
0,2026-08-15T09:31:08.472934+00:00,0241-001-101-016:1000,20,4,-108,True
1,2026-08-15T09:31:08.472934+00:00,0241-001-101-017:1000,2,0,425,False
2,2026-08-15T09:31:08.472934+00:00,0241-001-102-016:1000,17,3,-45,True
3,2026-08-15T09:31:08.472934+00:00,0241-001-102-017:1000,2,0,845,False
4,2026-08-15T09:31:08.472934+00:00,0241-001-103-016:1000,15,1,63,False


# Delay distribution

Delay is reported in seconds, so these buckets show how far from timetable the service actually runs.

In [28]:
print(current_df["arrival_delay"].describe())
pd.cut(current_df["arrival_delay"], [-10**9, -60, -1, 0, 60, 300, 10**9], labels=["early >1m", "early", "on time", "late <1m", "late 1-5m", "late >5m"]).value_counts().sort_index()

count    374.000000
mean       6.655080
std       11.555497
min      -25.000000
25%        0.000000
50%        0.000000
75%       12.000000
max       45.000000
Name: arrival_delay, dtype: float64


arrival_delay
early >1m      0
early         14
on time      175
late <1m     185
late 1-5m      0
late >5m       0
Name: count, dtype: int64

# Trips that have not departed yet

Trips with no passed stop fall back to their next stop, where the feed has usually measured nothing yet and reports close to zero delay.
These rows need watching since they pull the whole distribution towards zero.

In [29]:
print(current_df["is_passed"].value_counts())
current_df.groupby("is_passed")["arrival_delay"].agg(["count", "mean", "min", "max"])

is_passed
True     207
False    167
Name: count, dtype: int64


,count,mean,min,max
is_passed,,,,
False,167,0.371257,-25,21
True,207,11.724638,-25,45


In [30]:
current_df.groupby("stop_sequence")["arrival_delay"].agg(["count", "mean", "max"]).head(25)

,count,mean,max
stop_sequence,,,
2,159,0.000000,0
3,6,2.000000,5
4,4,4.000000,8
5,9,5.111111,26
6,4,22.000000,26
7,8,16.875000,24
8,11,8.636364,24
9,13,2.384615,18
10,6,-1.500000,2


# Vehicle movement

Speed, stop status and occupancy are the candidate features coming from the vehicle positions feed.

In [31]:
print(vehicle_positions_df["current_status"].value_counts())
print(vehicle_positions_df["occupancy_status"].value_counts())
print(vehicle_positions_df["vehicle_id"].nunique(), "vehicles")
vehicle_positions_df.groupby("current_status")["speed"].describe()

current_status
IN_TRANSIT_TO    122
STOPPED_AT        29
Name: count, dtype: int64
occupancy_status
MANY_SEATS_AVAILABLE    86
FEW_SEATS_AVAILABLE     59
STANDING_ROOM_ONLY       6
Name: count, dtype: int64
19 vehicles


,count,mean,std,min,25%,50%,75%,max
current_status,,,,,,,,
IN_TRANSIT_TO,122.0,19.090164,8.655963,0.0,14.0,21.5,27.0,28.0
STOPPED_AT,29.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


# Service alerts

Check how many distinct alerts there are and how many of them actually concern Sydney Metro routes.

In [32]:
print(alerts_df["effect"].value_counts())
print(alerts_df["cause"].value_counts())
print(alerts_df["entity_id"].nunique(), "distinct alerts across", alerts_df["route_id"].nunique(), "routes")
alerts_df["route_id"].str.startswith("SMNW").value_counts()

effect
UNKNOWN_EFFECT      199
MODIFIED_SERVICE      1
Name: count, dtype: int64
cause
UNKNOWN_CAUSE    199
MAINTENANCE        1
Name: count, dtype: int64
2 distinct alerts across 199 routes


route_id
False    198
True       2
Name: count, dtype: int64

# One-minute windows

Aggregate the per-trip delays and the vehicle state onto a fixed one-minute grid, which is the unit the model will predict on.

In [33]:
ON_TIME_THRESHOLD_SECONDS = 60

current_df["window_start"] = pd.to_datetime(current_df["fetched_at"], utc=True, format="ISO8601").dt.floor("1min")
vehicle_positions_df["window_start"] = pd.to_datetime(vehicle_positions_df["fetched_at"], utc=True, format="ISO8601").dt.floor("1min")

delay_windows = current_df.groupby("window_start").agg(
    mean_delay=("arrival_delay", "mean"),
    median_delay=("arrival_delay", "median"),
    max_delay=("arrival_delay", "max"),
    min_delay=("arrival_delay", "min"),
    p90_delay=("arrival_delay", lambda s: s.quantile(0.9)),
    frac_late=("arrival_delay", lambda s: (s > ON_TIME_THRESHOLD_SECONDS).mean()),
    n_trips=("trip_id", "nunique"),
    n_polls=("fetched_at", "nunique"),
)

vehicle_windows = vehicle_positions_df.groupby("window_start").agg(
    n_vehicles=("vehicle_id", "nunique"),
    mean_speed=("speed", "mean"),
    frac_stopped=("current_status", lambda s: (s == "STOPPED_AT").mean()),
)

windows_df = delay_windows.join(vehicle_windows, how="outer").sort_index()
windows_df

,mean_delay,median_delay,max_delay,min_delay,p90_delay,frac_late,n_trips,n_polls,n_vehicles,mean_speed,frac_stopped
window_start,,,,,,,,,,,
2026-08-15 09:31:00+00:00,5.976744,1.0,45,-3,20.4,0.0,43,1,17,14.294118,0.235294
2026-08-15 09:36:00+00:00,6.585366,1.0,33,-3,22.0,0.0,41,1,18,16.166667,0.222222
2026-08-15 09:38:00+00:00,6.372093,0.0,36,-14,25.2,0.0,43,1,17,14.117647,0.235294
2026-08-15 09:52:00+00:00,7.202381,1.0,36,-25,24.0,0.0,42,2,17,18.382353,0.117647
2026-08-15 09:55:00+00:00,6.585366,0.0,36,-25,24.0,0.0,41,2,16,14.187500,0.218750
2026-08-15 09:56:00+00:00,6.703704,0.0,36,-25,24.0,0.0,41,2,17,14.424242,0.181818


# How much the not-departed trips move each window

Rebuild the same windows from measured stops only, to see how far the fallback rows shift the mean.

In [34]:
measured_windows = current_df[current_df["is_passed"]].groupby("window_start")["arrival_delay"].agg(
    mean_delay_measured="mean", n_measured="size"
)
windows_df[["mean_delay", "n_trips"]].join(measured_windows)

,mean_delay,n_trips,mean_delay_measured,n_measured
window_start,,,,
2026-08-15 09:31:00+00:00,5.976744,43,11.130435,23
2026-08-15 09:36:00+00:00,6.585366,41,11.250000,24
2026-08-15 09:38:00+00:00,6.372093,43,11.416667,24
2026-08-15 09:52:00+00:00,7.202381,42,14.325581,43
2026-08-15 09:55:00+00:00,6.585366,41,10.695652,46
2026-08-15 09:56:00+00:00,6.703704,41,11.042553,47


# Targets at 5, 10 and 15 minutes

Polling is irregular, so the windows are reindexed onto a gapless grid first and a horizon then means minutes of wall clock rather than a number of polls.
Only one route appears in the feed, so this is a single series rather than one per route.

In [35]:
grid = windows_df.copy()
grid["is_observed"] = True
grid = grid.reindex(pd.date_range(grid.index.min(), grid.index.max(), freq="1min"))
grid["is_observed"] = grid["is_observed"].fillna(False).astype(bool)

# One row is one minute on this grid, so shifting back by the horizon reads the future window without any leakage.
target_cols = []
for horizon in (5, 10, 15):
    col = f"target_delay_{horizon}min"
    grid[col] = grid["mean_delay"].shift(-horizon)
    target_cols.append(col)

targets_df = grid.rename_axis("window_start").reset_index()
targets_df[["window_start", "is_observed", "mean_delay", *target_cols]]

,window_start,is_observed,mean_delay,target_delay_5min,target_delay_10min,target_delay_15min
0,2026-08-15 09:31:00+00:00,True,5.976744,6.585366,NaN,NaN
1,2026-08-15 09:32:00+00:00,False,NaN,NaN,NaN,NaN
2,2026-08-15 09:33:00+00:00,False,NaN,6.372093,NaN,NaN
3,2026-08-15 09:34:00+00:00,False,NaN,NaN,NaN,NaN
4,2026-08-15 09:35:00+00:00,False,NaN,NaN,NaN,NaN
5,2026-08-15 09:36:00+00:00,True,6.585366,NaN,NaN,NaN
6,2026-08-15 09:37:00+00:00,False,NaN,NaN,NaN,7.202381
7,2026-08-15 09:38:00+00:00,True,6.372093,NaN,NaN,NaN
8,2026-08-15 09:39:00+00:00,False,NaN,NaN,NaN,NaN
9,2026-08-15 09:40:00+00:00,False,NaN,NaN,NaN,6.585366


# How many rows are actually trainable

A row is only usable if the window was polled and its future window was polled too.

In [36]:
usable = targets_df[targets_df["is_observed"]]
print("observed windows:", len(usable), "of", len(targets_df))
usable[target_cols].notna().sum()

observed windows: 6 of 26


target_delay_5min     1
target_delay_10min    0
target_delay_15min    0
dtype: int64

# Predicting per trip instead of per route

The route window mean averages more than forty trips into one number, which leaves almost nothing to learn from.
Reshaping the same observations to one row per trip and window keeps the spread and multiplies the usable rows.

In [37]:
trip_delay_by_window = current_df.groupby(["trip_id", "window_start"], as_index=False)["arrival_delay"].mean()

# Shifting the lookup back by the horizon lets a left merge read the same trip's later delay, so nothing from the past is used.
trip_targets = trip_delay_by_window.copy()
trip_target_cols = []
for horizon in (5, 10, 15):
    col = f"target_delay_{horizon}min"
    future = trip_delay_by_window.rename(columns={"arrival_delay": col})
    future["window_start"] = future["window_start"] - pd.Timedelta(minutes=horizon)
    trip_targets = trip_targets.merge(future, on=["trip_id", "window_start"], how="left")
    trip_target_cols.append(col)

print(trip_targets.shape)
trip_targets.dropna(subset=["target_delay_5min"]).head()

(251, 6)


,trip_id,window_start,arrival_delay,target_delay_5min,target_delay_10min,target_delay_15min
0,0241-001-101-016:1000,2026-08-15 09:31:00+00:00,4.0,20.0,NaN,NaN
6,0241-001-101-017:1000,2026-08-15 09:31:00+00:00,0.0,0.0,NaN,NaN
16,0241-001-102-016:1000,2026-08-15 09:31:00+00:00,3.0,3.0,NaN,NaN
22,0241-001-102-017:1000,2026-08-15 09:31:00+00:00,0.0,0.0,NaN,NaN
28,0241-001-103-016:1000,2026-08-15 09:31:00+00:00,1.0,1.0,NaN,NaN


In [38]:
print(f"route window mean : {usable['target_delay_5min'].notna().sum()} rows with a 5 min target, predicted series std {windows_df['mean_delay'].std():.2f} s")
print(f"per trip          : {trip_targets['target_delay_5min'].notna().sum()} rows with a 5 min target, predicted series std {trip_delay_by_window['arrival_delay'].std():.2f} s")

trip_targets[trip_target_cols].notna().sum()

route window mean : 1 rows with a 5 min target, predicted series std 0.40 s
per trip          : 41 rows with a 5 min target, predicted series std 11.22 s


target_delay_5min     41
target_delay_10min     0
target_delay_15min     0
dtype: int64

# Findings

- Delays are tiny. Median is 0 seconds and the worst is 45, so no train is ever a full minute late and a 60 second late flag marks nothing.
- Only stops a train has already passed carry a real measurement. Everything further ahead is the operator's guess and sits near zero.
- Averaging all trips into one number per minute flattens the signal, leaving 4% of the spread. Predicting each trip separately keeps it and gives 41 usable rows instead of 1.
- Almost half the rows are trains that have not started yet, which report zero and drag the averages down.
- There is not enough data yet. 25 minutes of uneven polling gives nothing at all for the 10 and 15 minute horizons, so steady collection is the next thing needed.

# Next step

The windowing holds up and can be turned into scripts, with the window aggregates kept as features and the per-trip framing used for the target.
Two things to settle first: whether the not-departed trips are dropped, and how long a run of regular collection is needed before the 10 and 15 minute horizons produce any rows.